# TRAVI — GOOGLE COLAB ON-DEMAND GPU INFERENCE BACKEND
### Qwen2-7B-Instruct with Ollama, Vietnam Tourism RAG & Adaptive Personalization

This notebook establishes an on-demand, free GPU inference server for the **Travi Vietnam Travel Platform**.
- **Hardware Target:** Google Colab T4 GPU (15GB GDDR6 VRAM)
- **Model:** `qwen2:7b` (Qwen2-7B-Instruct Q4_0 quantized GGUF, ~4.5 GB VRAM)
- **Inference Engine:** Ollama daemon with persistent VRAM pinning (`keep_alive: -1`)
- **Backend:** FastAPI with Token Guard, Semaphore Concurrency Limiter (2), and Adaptive Customer Profiling
- **Tunnel:** Cloudflare Tunnel (`pycloudflared`) exposing a temporary authenticated HTTPS endpoint.

> [!IMPORTANT]
> **Temporary Server:** This inference endpoint remains active while this Colab session is running. When the Colab runtime disconnects or restarts, the tunnel expires, and Travi seamlessly continues in Verified Deterministic Expert mode.

In [ ]:
# STEP 1: VERIFY HARDWARE & GPU AVAILABILITY
# Rule: Verify actual CUDA GPU availability before loading the model. Do not assume a GPU was assigned.
import torch
import subprocess
import sys

print("=== CHECKING GOOGLE COLAB CUDA GPU RUNTIME ===")
if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected. Please go to Colab Menu -> Runtime -> Change runtime type -> Select T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024))
print(f"SUCCESS: Detected {gpu_name} with {vram_mb} MB VRAM.")
subprocess.run(["nvidia-smi"])


In [ ]:
# STEP 2: INSTALL DEPENDENCIES & OLLAMA SERVER
!pip install -q fastapi uvicorn httpx pydantic pycloudflared nest-asyncio kagglehub

# Install official Ollama binary on Linux
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# STEP 3: START OLLAMA DAEMON IN BACKGROUND
# Note: Ollama binds strictly to 127.0.0.1:11434 and is NEVER exposed directly
import subprocess
import time
import httpx

print("Starting Ollama daemon on 127.0.0.1:11434...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for Ollama to become ready
for attempt in range(20):
    try:
        resp = httpx.get("http://127.0.0.1:11434/api/tags", timeout=2.0)
        if resp.status_code == 200:
            print("Ollama daemon is healthy and ready.")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start within 20 seconds.")


In [ ]:
# STEP 5: WRITE AUDITED & HARDENED FASTAPI BACKEND APPLICATION SCRIPT (app.py)
# Hardened CORS, 16KB body size limit, prompt injection defense, and multi-dimensional RAG
with open("app.py", "w", encoding="utf-8") as f:
    f.write("\"\"\"\nTRAVI AI ASSISTANT \u2014 GOOGLE COLAB GPU INFERENCE BACKEND (Qwen2-7B-Instruct)\n=============================================================================\nFastAPI Service with Ollama backend, In-memory Vietnam Tourism RAG,\nAdaptive Customer Profile Personalization, Token Authentication,\nConcurrency Semaphore, and Streaming SSE support.\n\nHardware Target: Google Colab T4 GPU (15GB GDDR6 VRAM) or Tesla P100\nModel: Qwen2-7B-Instruct (Q4_0 Quantized GGUF, ~4.5GB VRAM)\n\"\"\"\n\nimport os\nimport sys\nimport json\nimport time\nimport asyncio\nimport logging\nfrom typing import List, Optional, Dict, Any\nfrom contextlib import asynccontextmanager\n\nfrom fastapi import FastAPI, Header, HTTPException, Depends, Request\nfrom fastapi.responses import StreamingResponse, JSONResponse\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel, Field\nimport httpx\n\nimport secrets\n\n# Configure logging\nlogging.basicConfig(\n    level=logging.INFO,\n    format=\"%(asctime)s [%(levelname)s] %(name)s: %(message)s\"\n)\nlogger = logging.getLogger(\"travi-colab-gpu\")\n\n# Configuration from environment: NEVER use a predictable hardcoded default\nTOKEN_FILE_PATH = \"/content/.travi_token\" if os.path.exists(\"/content\") else \".travi_token\"\n\ndef resolve_api_token() -> str:\n    \"\"\"Safely retrieves token from env or creates a 600-permission session token file.\"\"\"\n    env_token = os.getenv(\"COLAB_AI_API_TOKEN\") or os.getenv(\"TRAVI_COLAB_API_KEY\")\n    if env_token and env_token.strip():\n        return env_token.strip()\n\n    if os.path.exists(TOKEN_FILE_PATH):\n        try:\n            with open(TOKEN_FILE_PATH, \"r\", encoding=\"utf-8\") as f:\n                token = f.read().strip()\n                if token:\n                    return token\n        except Exception:\n            pass\n\n    generated = secrets.token_urlsafe(24)\n    try:\n        with open(TOKEN_FILE_PATH, \"w\", encoding=\"utf-8\") as f:\n            f.write(generated)\n        try:\n            os.chmod(TOKEN_FILE_PATH, 0o600)\n        except Exception:\n            pass\n    except Exception:\n        pass\n    return generated\n\nCOLAB_AI_API_TOKEN = resolve_api_token()\nlogger.info(\"Authentication security initialized with cryptographically secure token.\")\n\nOLLAMA_BASE_URL = os.getenv(\"OLLAMA_BASE_URL\", \"http://127.0.0.1:11434\")\nMODEL_NAME = os.getenv(\"MODEL_NAME\", \"qwen2:7b\")\nMAX_CONCURRENT_REQUESTS = int(os.getenv(\"MAX_CONCURRENT_REQUESTS\", \"2\"))\n# Colab Ollama timeout: 35s to respect hierarchy (Ollama 35s < Gateway 45s < Cloud Functions 60s)\nREQUEST_TIMEOUT_SECONDS = float(os.getenv(\"REQUEST_TIMEOUT_SECONDS\", \"35.0\"))\nMAX_REQUEST_BODY_BYTES = 16 * 1024  # 16KB max payload limit\n\n# Concurrency limiter to protect 15GB/16GB VRAM from thrashing\ninference_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)\n\n# GPU inspection state\ngpu_info: Dict[str, Any] = {\n    \"detected\": False,\n    \"name\": \"Unknown\",\n    \"total_memory_mb\": 0,\n    \"cuda_available\": False,\n}\n\ndef verify_gpu_environment():\n    \"\"\"Verify actual CUDA GPU availability on Google Colab.\"\"\"\n    global gpu_info\n    try:\n        import torch\n        if torch.cuda.is_available():\n            gpu_count = torch.cuda.device_count()\n            device_name = torch.cuda.get_device_name(0)\n            vram_bytes = torch.cuda.get_device_properties(0).total_memory\n            vram_mb = int(vram_bytes / (1024 * 1024))\n            gpu_info = {\n                \"detected\": True,\n                \"cuda_available\": True,\n                \"device_count\": gpu_count,\n                \"name\": device_name,\n                \"total_memory_mb\": vram_mb,\n            }\n            logger.info(f\"Verified GPU: {device_name} ({vram_mb} MB VRAM)\")\n        else:\n            gpu_info = {\n                \"detected\": False,\n                \"cuda_available\": False,\n                \"name\": \"No CUDA device found\",\n                \"total_memory_mb\": 0,\n            }\n            logger.warning(\"No CUDA GPU detected! Google Colab runtime is running in CPU mode.\")\n    except Exception as e:\n        logger.error(f\"Failed to inspect GPU with PyTorch: {e}\")\n        gpu_info[\"error\"] = str(e)\n\n# In-memory Tourism RAG Corpus\nrag_corpus: List[Dict[str, Any]] = []\n\ndef load_embedded_rag_corpus():\n    \"\"\"Loads verified Vietnam POIs and Cultural Assets into memory for grounding.\"\"\"\n    global rag_corpus\n\n    default_corpus = [\n        {\n            \"id\": \"son-la-moc-chau\",\n            \"name\": \"Khu du l\u1ecbch Qu\u1ed1c gia M\u1ed9c Ch\u00e2u\",\n            \"province\": \"S\u01a1n La\",\n            \"subregion\": \"T\u00e2y B\u1eafc\",\n            \"category\": \"Khu du l\u1ecbch qu\u1ed1c gia\",\n            \"description\": \"Cao nguy\u00ean M\u1ed9c Ch\u00e2u n\u1ed5i ti\u1ebfng v\u1edbi \u0111\u1ed3i ch\u00e8 tr\u00e1i tim, thung l\u0169ng m\u1eadn N\u00e0 Ka, r\u1eebng th\u00f4ng b\u1ea3n \u00c1ng v\u00e0 th\u00e1c D\u1ea3i Y\u1ebfm. Kh\u00ed h\u1eadu m\u00e1t m\u1ebb quanh n\u0103m.\",\n            \"experiences\": [\"Ng\u1eafm hoa m\u1eadn\", \"H\u00e1i ch\u00e8\", \"U\u1ed1ng s\u1eefa t\u01b0\u01a1i M\u1ed9c Ch\u00e2u\", \"Kh\u00e1m ph\u00e1 b\u1ea3n l\u00e0ng Th\u00e1i\"],\n            \"foods\": [\"B\u00ea chao M\u1ed9c Ch\u00e2u\", \"C\u00e1 su\u1ed1i n\u01b0\u1edbng Pa P\u1ec9nh T\u1ed9p\", \"Th\u1ecbt tr\u00e2u g\u00e1c b\u1ebfp\", \"X\u00f4i ng\u0169 s\u1eafc\"],\n            \"source\": \"S\u1edf V\u0103n h\u00f3a Th\u1ec3 thao v\u00e0 Du l\u1ecbch t\u1ec9nh S\u01a1n La\",\n            \"coordinate_status\": \"unverified_field_notes\",\n            \"weather_notes\": \"M\u00f9a xu\u00e2n (Th\u00e1ng 1-2) hoa m\u1eadn n\u1edf tr\u1eafng; M\u00f9a h\u00e8 m\u00e1t m\u1ebb d\u1ec5 ch\u1ecbu; M\u00f9a \u0111\u00f4ng se l\u1ea1nh nhi\u1ec1u s\u01b0\u01a1ng m\u00f9.\"\n        },\n        {\n            \"id\": \"son-la-hang-a-phu\",\n            \"name\": \"\u0110i\u1ec3m du l\u1ecbch Hang v\u1ee3 ch\u1ed3ng A Ph\u1ee7\",\n            \"province\": \"S\u01a1n La\",\n            \"subregion\": \"T\u00e2y B\u1eafc\",\n            \"category\": \"\u0110i\u1ec3m du l\u1ecbch\",\n            \"description\": \"N\u1eb1m t\u1ea1i b\u1ea3n H\u1ed3ng Ng\u00e0i, x\u00e3 B\u1eafc Y\u00ean. Hang A Ph\u1ee7 c\u00f3 c\u1ea3nh quan \u0111\u1ecba ch\u1ea5t \u0111\u1ed9c \u0111\u00e1o, g\u1eafn li\u1ec1n v\u1edbi t\u00e1c ph\u1ea9m v\u0103n h\u1ecdc V\u1ee3 ch\u1ed3ng A Ph\u1ee7 c\u1ee7a T\u00f4 Ho\u00e0i.\",\n            \"experiences\": [\"Tham quan hang \u0111\u1ed9ng\", \"T\u00ecm hi\u1ec3u v\u0103n h\u00f3a d\u00e2n t\u1ed9c H'M\u00f4ng\"],\n            \"foods\": [\"Th\u1ecbt tr\u00e2u s\u1ea5y\", \"R\u01b0\u1ee3u ng\u00f4 B\u1eafc Y\u00ean\", \"G\u00e0 \u0111en b\u1ea3n \u0111\u1ecba\"],\n            \"source\": \"S\u1edf V\u0103n h\u00f3a Th\u1ec3 thao v\u00e0 Du l\u1ecbch t\u1ec9nh S\u01a1n La\",\n            \"coordinate_status\": \"unverified_field_notes\",\n            \"weather_notes\": \"V\u00f9ng n\u00fai cao B\u1eafc Y\u00ean c\u00f3 s\u01b0\u01a1ng m\u00f9 d\u00e0y v\u00e0o m\u00f9a \u0111\u00f4ng; m\u00f9a m\u01b0a t\u1eeb th\u00e1ng 6-8 c\u1ea7n ch\u00fa \u00fd tr\u01a1n tr\u01b0\u1ee3t.\"\n        },\n        {\n            \"id\": \"son-la-ta-xua\",\n            \"name\": \"Khu du l\u1ecbch T\u00e0 X\u00f9a - Thi\u00ean \u0111\u01b0\u1eddng m\u00e2y\",\n            \"province\": \"S\u01a1n La\",\n            \"subregion\": \"T\u00e2y B\u1eafc\",\n            \"category\": \"Danh lam th\u1eafng c\u1ea3nh\",\n            \"description\": \"D\u00e3y n\u00fai T\u00e0 X\u00f9a gi\u00e1p ranh gi\u1eefa S\u01a1n La v\u00e0 Y\u00ean B\u00e1i, n\u1ed5i ti\u1ebfng v\u1edbi s\u1ed1ng l\u01b0ng kh\u1ee7ng long H\u00e1ng \u0110\u1ed3ng v\u00e0 bi\u1ec3n m\u00e2y b\u1ed3ng b\u1ec1nh h\u00f9ng v\u0129.\",\n            \"experiences\": [\"S\u0103n m\u00e2y T\u00e0 X\u00f9a\", \"Trekking s\u1ed1ng l\u01b0ng kh\u1ee7ng long\", \"Th\u01b0\u1edfng th\u1ee9c ch\u00e8 Shan tuy\u1ebft c\u1ed5 th\u1ee5\"],\n            \"foods\": [\"Th\u1ecbt l\u1ee3n b\u1ea3n n\u01b0\u1edbng\", \"M\u0103ng tr\u00fac x\u00e0o\", \"C\u01a1m lam\"],\n            \"source\": \"S\u1edf V\u0103n h\u00f3a Th\u1ec3 thao v\u00e0 Du l\u1ecbch t\u1ec9nh S\u01a1n La\",\n            \"coordinate_status\": \"unverified_field_notes\",\n            \"weather_notes\": \"M\u00f9a s\u0103n m\u00e2y \u0111\u1eb9p nh\u1ea5t t\u1eeb th\u00e1ng 10 \u0111\u1ebfn th\u00e1ng 4 n\u0103m sau; nhi\u1ec7t \u0111\u1ed9 ban \u0111\u00eam c\u00f3 th\u1ec3 xu\u1ed1ng d\u01b0\u1edbi 10 \u0111\u1ed9 C.\"\n        },\n        {\n            \"id\": \"cultural-trong-dong-ngoc-lu\",\n            \"name\": \"Tr\u1ed1ng \u0111\u1ed3ng Ng\u1ecdc L\u0169\",\n            \"province\": \"H\u00e0 N\u1ed9i (L\u01b0u gi\u1eef) / H\u00e0 Nam (Ph\u00e1t hi\u1ec7n)\",\n            \"subregion\": \"\u0110\u1ed3ng b\u1eb1ng s\u00f4ng H\u1ed3ng\",\n            \"category\": \"B\u1ea3o v\u1eadt qu\u1ed1c gia (Hi\u1ec7n v\u1eadt v\u0103n h\u00f3a)\",\n            \"description\": \"B\u1ea3o v\u1eadt qu\u1ed1c gia s\u1ed1 1 c\u1ee7a Vi\u1ec7t Nam thu\u1ed9c V\u0103n h\u00f3a \u0110\u00f4ng S\u01a1n, ni\u00ean \u0111\u1ea1i th\u1ebf k\u1ef7 II-III TCN. M\u1eb7t tr\u1ed1ng kh\u1eafc h\u1ecda sao 14 c\u00e1nh, chim L\u1ea1c v\u00e0 c\u1ea3nh sinh ho\u1ea1t th\u1eddi H\u00f9ng V\u01b0\u01a1ng. Hi\u1ec7n v\u1eadt \u0111\u01b0\u1ee3c l\u01b0u gi\u1eef t\u1ea1i B\u1ea3o t\u00e0ng L\u1ecbch s\u1eed Qu\u1ed1c gia (S\u1ed1 1 Tr\u00e0ng Ti\u1ec1n, H\u00e0 N\u1ed9i).\",\n            \"experiences\": [\"Tham quan gian tr\u01b0ng b\u00e0y Kim kh\u00ed th\u1eddi H\u00f9ng V\u01b0\u01a1ng\", \"T\u00ecm hi\u1ec3u k\u1ef9 thu\u1eadt \u0111\u00fac \u0111\u1ed3ng \u0110\u00f4ng S\u01a1n\"],\n            \"foods\": [\"\u1ea8m th\u1ef1c ph\u1ed1 c\u1ed5 H\u00e0 N\u1ed9i\", \"Ch\u1ea3 c\u00e1 L\u00e3 V\u1ecdng\", \"C\u00e0 ph\u00ea tr\u1ee9ng\"],\n            \"source\": \"B\u1ea3o t\u00e0ng L\u1ecbch s\u1eed Qu\u1ed1c gia & C\u1ee5c Di s\u1ea3n V\u0103n h\u00f3a\",\n            \"coordinate_status\": \"museum_venue_verified\",\n            \"weather_notes\": \"B\u1ea3o t\u00e0ng m\u1edf c\u1eeda quanh n\u0103m trong ph\u00f2ng \u0111i\u1ec1u h\u00f2a; H\u00e0 N\u1ed9i \u0111\u1eb9p nh\u1ea5t v\u00e0o m\u00f9a thu th\u00e1ng 9-11.\"\n        },\n        {\n            \"id\": \"cultural-xoe-thai\",\n            \"name\": \"Ngh\u1ec7 thu\u1eadt X\u00f2e Th\u00e1i\",\n            \"province\": \"S\u01a1n La, Y\u00ean B\u00e1i, \u0110i\u1ec7n Bi\u00ean, Lai Ch\u00e2u\",\n            \"subregion\": \"T\u00e2y B\u1eafc\",\n            \"category\": \"Di s\u1ea3n v\u0103n h\u00f3a phi v\u1eadt th\u1ec3 \u0111\u1ea1i di\u1ec7n nh\u00e2n lo\u1ea1i (UNESCO)\",\n            \"description\": \"\u0110i\u1ec7u m\u00faa d\u00e2n gian truy\u1ec1n th\u1ed1ng bi\u1ec3u tr\u01b0ng cho tinh th\u1ea7n c\u1ed9ng \u0111\u1ed3ng, l\u00f2ng hi\u1ebfu kh\u00e1ch c\u1ee7a ng\u01b0\u1eddi Th\u00e1i T\u00e2y B\u1eafc. G\u1ed3m c\u00e1c \u0111i\u1ec7u m\u00faa kh\u0103n, x\u00f2e v\u00f2ng b\u00ean b\u1ebfp l\u1eeda b\u1ea3n l\u00e0ng.\",\n            \"experiences\": [\"Tham gia v\u00f2ng x\u00f2e hoa t\u1ea1i B\u1ea3n \u00c1ng ho\u1eb7c B\u1ea3n V\u1eb7t\", \"Giao l\u01b0u v\u0103n ngh\u1ec7 d\u00e2n gian\"],\n            \"foods\": [\"C\u01a1m lam\", \"C\u00e1 n\u01b0\u1edbng Pa P\u1ec9nh T\u1ed9p\", \"R\u01b0\u1ee3u c\u1ea7n\"],\n            \"source\": \"UNESCO & S\u1edf VHTT&DL c\u00e1c t\u1ec9nh T\u00e2y B\u1eafc\",\n            \"coordinate_status\": \"intangible_cultural_heritage\",\n            \"weather_notes\": \"C\u00e1c l\u1ec5 h\u1ed9i di\u1ec5n ra r\u1ed9n r\u00e0ng v\u00e0o d\u1ecbp \u0111\u1ea7u xu\u00e2n v\u00e0 m\u00f9a g\u1eb7t l\u00faa ch\u00edn.\"\n        }\n    ]\n\n    # Check if dataset files exist in Colab environment (e.g. kagglehub or uploaded CSV)\n    csv_paths = [\n        \"son_la_tourism_normalized_utf8.csv\",\n        \"vietnam_tourism_starter.csv\",\n        \"/content/son_la_tourism_normalized_utf8.csv\",\n        \"/content/vietnam_tourism_starter.csv\"\n    ]\n    for p in csv_paths:\n        if os.path.exists(p):\n            try:\n                import csv\n                with open(p, mode=\"r\", encoding=\"utf-8-sig\") as f:\n                    reader = csv.DictReader(f)\n                    count = 0\n                    for row in reader:\n                        name = row.get(\"name\") or row.get(\"ten_diem\") or row.get(\"title\")\n                        if name:\n                            default_corpus.append({\n                                \"id\": row.get(\"id\") or f\"csv-{count}\",\n                                \"name\": name,\n                                \"province\": row.get(\"province\") or row.get(\"tinh_thanh\") or \"Vi\u1ec7t Nam\",\n                                \"subregion\": row.get(\"subregion\") or \"B\u1eafc B\u1ed9\",\n                                \"category\": row.get(\"category\") or \"\u0110i\u1ec3m tham quan\",\n                                \"description\": row.get(\"description\") or row.get(\"mo_ta\") or \"\",\n                                \"experiences\": [row.get(\"experiences\")] if row.get(\"experiences\") else [\"Kh\u00e1m ph\u00e1 \u0111\u1ecba ph\u01b0\u01a1ng\"],\n                                \"foods\": [row.get(\"foods\")] if row.get(\"foods\") else [\"\u1ea8m th\u1ef1c b\u1ea3n \u0111\u1ecba\"],\n                                \"source\": row.get(\"source\") or \"B\u1ed9 d\u1eef li\u1ec7u Du l\u1ecbch Vi\u1ec7t Nam\",\n                                \"coordinate_status\": \"needs_review\",\n                                \"weather_notes\": \"Tham kh\u1ea3o d\u1ef1 b\u00e1o th\u1eddi ti\u1ebft \u0111\u1ecba ph\u01b0\u01a1ng tr\u01b0\u1edbc khi kh\u1edfi h\u00e0nh.\"\n                            })\n                            count += 1\n                logger.info(f\"Loaded {count} additional records from CSV dataset: {p}\")\n            except Exception as ex:\n                logger.warning(f\"Could not read CSV {p}: {ex}\")\n\n    rag_corpus = default_corpus\n    logger.info(f\"Loaded {len(rag_corpus)} core grounding facts into RAG retrieval memory.\")\n\ndef retrieve_rag_context(query: str, top_k: int = 3) -> List[Dict[str, Any]]:\n    \"\"\"Semantic token-matching retriever for zero-latency RAG grounding.\"\"\"\n    q_lower = query.lower()\n    scored = []\n    for item in rag_corpus:\n        score = 0\n        searchable_text = f\"{item['name']} {item['description']} {' '.join(item.get('experiences', []))} {' '.join(item.get('foods', []))} {item['province']} {item.get('subregion', '')}\".lower()\n        for token in q_lower.split():\n            if len(token) > 2 and token in searchable_text:\n                score += 1\n        if item['name'].lower() in q_lower:\n            score += 6\n        if item['province'].lower() in q_lower:\n            score += 3\n        if score > 0:\n            scored.append((score, item))\n\n    scored.sort(key=lambda x: x[0], reverse=True)\n    return [item for _, item in scored[:top_k]]\n\n# Lifespan: warm up model in Ollama\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    verify_gpu_environment()\n    load_embedded_rag_corpus()\n    # Check Ollama connectivity & keep model loaded in VRAM\n    async with httpx.AsyncClient(timeout=15.0) as client:\n        try:\n            resp = await client.get(f\"{OLLAMA_BASE_URL}/api/tags\")\n            if resp.status_code == 200:\n                logger.info(f\"Ollama server connected at {OLLAMA_BASE_URL}.\")\n                warmup_payload = {\n                    \"model\": MODEL_NAME,\n                    \"prompt\": \"Xin ch\u00e0o, tr\u1ee3 l\u00fd du l\u1ecbch Vi\u1ec7t Nam \u0111\u00e3 s\u1eb5n s\u00e0ng.\",\n                    \"keep_alive\": -1,\n                    \"stream\": False\n                }\n                logger.info(f\"Warming up {MODEL_NAME} with keep_alive: -1...\")\n                warmup_resp = await client.post(f\"{OLLAMA_BASE_URL}/api/generate\", json=warmup_payload, timeout=60.0)\n                if warmup_resp.status_code == 200:\n                    logger.info(f\"Model {MODEL_NAME} successfully loaded and pinned in VRAM.\")\n            else:\n                logger.warning(f\"Ollama responded with status {resp.status_code}\")\n        except Exception as e:\n            logger.error(f\"Could not connect to Ollama: {e}. Ensure Ollama is running.\")\n    yield\n    logger.info(\"FastAPI Colab service shutting down.\")\n\napp = FastAPI(\n    title=\"Travi AI Assistant \u2014 Google Colab GPU Backend\",\n    description=\"On-demand GPU inference runtime serving Qwen2-7B-Instruct with Vietnam Tourism RAG & Adaptive Personalization\",\n    version=\"1.2.0\",\n    lifespan=lifespan\n)\n\n# 16KB Request Payload Guard Middleware\n@app.middleware(\"http\")\nasync def enforce_payload_limit(request: Request, call_next):\n    content_length = request.headers.get(\"content-length\")\n    if content_length:\n        try:\n            if int(content_length) > MAX_REQUEST_BODY_BYTES:\n                return JSONResponse(\n                    status_code=413,\n                    content={\"error\": \"Payload Too Large: Maximum allowed request size is 16KB.\", \"code\": \"PAYLOAD_TOO_LARGE\"}\n                )\n        except ValueError:\n            pass\n\n    # Read body safely to check chunked requests\n    body = await request.body()\n    if len(body) > MAX_REQUEST_BODY_BYTES:\n        return JSONResponse(\n            status_code=413,\n            content={\"error\": \"Payload Too Large: Maximum allowed request size is 16KB.\", \"code\": \"PAYLOAD_TOO_LARGE\"}\n        )\n\n    return await call_next(request)\n\n# Hardened CORS configuration (Zero wildcard * with credentials)\nALLOWED_ORIGINS = [\n    \"http://localhost:5173\",\n    \"http://127.0.0.1:5173\",\n    \"http://localhost:3000\",\n    \"http://localhost:3001\",\n    \"https://travi-app-23th10.web.app\",\n    \"https://travi-app-23th10.firebaseapp.com\",\n]\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=ALLOWED_ORIGINS,\n    allow_credentials=False,\n    allow_methods=[\"GET\", \"POST\", \"OPTIONS\"],\n    allow_headers=[\"*\"],\n)\n\n# Authentication Dependency with Constant-Time Verification\ndef verify_token(\n    authorization: Optional[str] = Header(None),\n    x_api_key: Optional[str] = Header(None)\n):\n    token = None\n    if authorization and authorization.startswith(\"Bearer \"):\n        token = authorization.replace(\"Bearer \", \"\").strip()\n    elif x_api_key:\n        token = x_api_key.strip()\n\n    if not token or not secrets.compare_digest(token, COLAB_AI_API_TOKEN):\n        raise HTTPException(\n            status_code=401,\n            detail=\"Unauthorized: Invalid or missing API authentication token.\",\n            headers={\"WWW-Authenticate\": \"Bearer\"}\n        )\n    return token\n\n# Request & Response Schemas\nclass LocationInfo(BaseModel):\n    country: Optional[str] = \"Vi\u1ec7t Nam\"\n    province: Optional[str] = None\n    city: Optional[str] = None\n\nclass CustomerProfile(BaseModel):\n    # Distinct coexisting dimensions:\n    market: Optional[str] = Field(default=None, description=\"DOMESTIC | INTERNATIONAL\")\n    origin_location: Optional[LocationInfo] = None\n    destination_location: Optional[LocationInfo] = None\n    travel_scope: Optional[str] = Field(default=None, description=\"LOCAL | INTERPROVINCIAL | INTERNATIONAL_TRAVEL\")\n    persona: Optional[str] = Field(default=None, description=\"FAMILY | BACKPACKER | SOLO | EXPLORER\")\n    language: Optional[str] = Field(default=None, description=\"VI | EN | OTHER\")\n\n    # Legacy & convenience fields\n    customer_type: Optional[str] = Field(default=None, description=\"Legacy alias: INTERNATIONAL | DOMESTIC | LOCAL | INTERPROVINCIAL\")\n    origin_country: Optional[str] = Field(default=\"VN\")\n    origin_province: Optional[str] = None\n    destination: Optional[str] = None\n    preferred_language: Optional[str] = Field(default=\"vi\")\n    budget: Optional[str] = Field(default=\"moderate\")\n    travel_duration: Optional[int] = Field(default=3)\n    interests: Optional[List[str]] = Field(default=[])\n    transportation_preference: Optional[str] = None\n\nclass ChatMessage(BaseModel):\n    role: str = Field(..., description=\"'user', 'assistant', or 'system'\")\n    content: str = Field(..., min_length=1, max_length=1000)\n\nclass ChatRequest(BaseModel):\n    prompt: str = Field(..., min_length=1, max_length=1000, description=\"User travel query\")\n    profile: Optional[CustomerProfile] = None\n    context: Optional[Dict[str, Any]] = None\n    history: Optional[List[ChatMessage]] = Field(default=[], max_length=10, description=\"Multi-turn conversation history (max 10 turns)\")\n    stream: Optional[bool] = Field(default=False, description=\"Enable Server-Sent Events streaming\")\n    # Prompt injection defense: client-side system overrides are rejected / ignored\n\nclass ModelInfoResponse(BaseModel):\n    model: str\n    quantization: str\n    gpu: Dict[str, Any]\n    keep_alive: str\n    rag_facts_count: int\n    context_window: int\n    runtime_expires_note: str\n    active_model_ready: bool = True\n    concurrency_max: int = MAX_CONCURRENT_REQUESTS\n\ndef build_adaptive_system_prompt(profile: Optional[CustomerProfile], rag_facts_text: str) -> str:\n    \"\"\"\n    Constructs prompt customized to multi-dimensional customer attributes:\n    - Market: DOMESTIC / INTERNATIONAL\n    - Travel Scope: LOCAL / INTERPROVINCIAL / INTERNATIONAL_TRAVEL\n    - Persona: FAMILY / BACKPACKER / SOLO / EXPLORER\n    - Language: VI / EN / OTHER\n    All dimensions coexist without conflicting or stereotyping.\n    \"\"\"\n    is_english = False\n    if profile:\n        if profile.language and profile.language.upper() == \"EN\":\n            is_english = True\n        elif profile.preferred_language and profile.preferred_language.lower().startswith(\"en\"):\n            is_english = True\n\n    if is_english:\n        base_instructions = (\n            \"You are Travi AI \u2014 a dedicated Vietnam travel and cultural intelligence assistant.\\n\"\n            \"DATA INTEGRITY & ETHICAL TOURISM PRINCIPLES (MANDATORY):\\n\"\n            \"1. ONLY state verified facts. NEVER fabricate ticket prices, opening hours, or assumed coordinates.\\n\"\n            \"2. Distinguish clearly: Cultural artifacts kept in museums differ from archaeological discovery sites; artifact coordinates are not museum entrance gates.\\n\"\n            \"3. If a site's coordinates or status have unverified field notes, advise travelers to confirm with local management before traveling.\\n\"\n            \"4. Respond in polite, engaging, and culturally evocative English while preserving authentic Vietnamese place and dish names.\"\n        )\n    else:\n        base_instructions = (\n            \"B\u1ea1n l\u00e0 Travi AI \u2014 Tr\u1ee3 l\u00fd chuy\u00ean gia du l\u1ecbch & v\u0103n h\u00f3a Vi\u1ec7t Nam t\u1eadn t\u00e2m.\\n\"\n            \"NGUY\u00caN T\u1eaeC TO\u00c0N V\u1eb8N D\u1eee LI\u1ec6U & \u0110\u1ea0O \u0110\u1ee8C DU L\u1ecaCH (B\u1eaeT BU\u1ed8C TU\u00c2N TH\u1ee6):\\n\"\n            \"1. CH\u1ec8 s\u1eed d\u1ee5ng s\u1ef1 th\u1eadt \u0111\u00e3 x\u00e1c minh. TUY\u1ec6T \u0110\u1ed0I KH\u00d4NG t\u1ef1 t\u1ea1o ra gi\u00e1 v\u00e9, gi\u1edd m\u1edf c\u1eeda ho\u1eb7c t\u1ecda \u0111\u1ed9 gi\u1ea3 \u0111\u1ecbnh.\\n\"\n            \"2. Ph\u00e2n bi\u1ec7t r\u00f5: Hi\u1ec7n v\u1eadt v\u0103n h\u00f3a (B\u1ea3o t\u00e0ng l\u01b0u gi\u1eef) KH\u00c1C v\u1edbi \u0111\u1ecba \u0111i\u1ec3m kh\u1ea3o c\u1ed5 n\u01a1i ph\u00e1t hi\u1ec7n; t\u1ecda \u0111\u1ed9 hi\u1ec7n v\u1eadt kh\u00f4ng ph\u1ea3i c\u1ed5ng b\u1ea3o t\u00e0ng.\\n\"\n            \"3. N\u1ebfu th\u00f4ng tin ho\u1eb7c t\u1ecda \u0111\u1ed9 c\u1ee7a \u0111i\u1ec3m \u0111\u1ebfn ch\u01b0a \u0111\u01b0\u1ee3c \u0111\u1ed1i so\u00e1t th\u1ef1c \u0111\u1ecba, h\u00e3y nh\u1eafc nh\u1edf du kh\u00e1ch li\u00ean h\u1ec7 ban qu\u1ea3n l\u00fd \u0111\u1ecba ph\u01b0\u01a1ng tr\u01b0\u1edbc khi di chuy\u1ec3n.\\n\"\n            \"4. Tr\u1ea3 l\u1eddi b\u1eb1ng ti\u1ebfng Vi\u1ec7t l\u1ecbch s\u1ef1, truy\u1ec1n c\u1ea3m, c\u00f3 chi\u1ec1u s\u00e2u v\u0103n h\u00f3a b\u1ea3n \u0111\u1ecba.\"\n        )\n\n    guidance_sections = []\n\n    # 1. Market Dimension (Domestic vs International)\n    market = \"DOMESTIC\"\n    if profile:\n        if profile.market:\n            market = profile.market.upper()\n        elif profile.customer_type == \"INTERNATIONAL\" or (profile.origin_country and profile.origin_country.lower() not in [\"vn\", \"vietnam\", \"vi\u1ec7t nam\"]):\n            market = \"INTERNATIONAL\"\n\n    if market == \"INTERNATIONAL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [International Market]: Provide deep cultural context, explain local customs, \"\n                \"temple & pagoda etiquette, VND currency conversion, e-visa/sim card advice, and food safety.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Th\u1ecb tr\u01b0\u1eddng Qu\u1ed1c t\u1ebf]: Cung c\u1ea5p h\u01b0\u1edbng d\u1eabn v\u0103n h\u00f3a s\u00e2u s\u1eafc, gi\u1ea3i th\u00edch phong t\u1ee5c t\u1eadp qu\u00e1n, \"\n                \"nghi th\u1ee9c \u0111\u1ec1n ch\u00f9a, quy \u0111\u1ed5i ti\u1ec1n t\u1ec7 VND, sim/data v\u00e0 an to\u00e0n th\u1ef1c ph\u1ea9m.\"\n            )\n    else:\n        if is_english:\n            guidance_sections.append(\n                \"- [Domestic Market]: Emphasize authentic local cuisine, cost-effective itineraries, and prominent landmarks.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Th\u1ecb tr\u01b0\u1eddng N\u1ed9i \u0111\u1ecba]: Ch\u00fa tr\u1ecdng \u1ea9m th\u1ef1c chu\u1ea9n v\u1ecb \u0111\u1ecba ph\u01b0\u01a1ng, chi ph\u00ed h\u1ee3p l\u00fd v\u00e0 \u0111\u1ecba \u0111i\u1ec3m n\u1ed5i ti\u1ebfng.\"\n            )\n\n    # 2. Travel Scope Dimension (Local vs Interprovincial vs Cross-Border)\n    scope = None\n    if profile:\n        scope = profile.travel_scope\n        if not scope and profile.customer_type:\n            if profile.customer_type in [\"LOCAL\", \"INTERPROVINCIAL\", \"INTERNATIONAL_TRAVEL\"]:\n                scope = profile.customer_type\n\n    if scope == \"LOCAL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Local Scope]: Focus on quiet hidden gems, humble local eateries, avoiding overcrowded tourist traps; suggest weekend getaways.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ph\u1ea1m vi B\u1ea3n \u0111\u1ecba]: T\u1eadp trung v\u00e0o \u0111i\u1ec3m \u0111\u1ebfn ng\u00e1ch (hidden gems), qu\u00e1n \u0103n d\u00e2n d\u00e3 \u00edt ng\u01b0\u1eddi bi\u1ebft, \"\n                \"tr\u00e1nh t\u1ee5 \u0111i\u1ec3m \u0111\u00f4ng \u0111\u00fac du l\u1ecbch \u0111\u1ea1i tr\u00e0; g\u1ee3i \u00fd d\u00e3 ngo\u1ea1i ng\u1eafn cu\u1ed1i tu\u1ea7n.\"\n            )\n    elif scope == \"INTERPROVINCIAL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Interprovincial Scope]: Optimize transportation routes, mountain pass conditions, travel duration, \"\n                \"and cultural connections between adjoining provinces.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ph\u1ea1m vi Li\u00ean t\u1ec9nh]: T\u1ed1i \u01b0u h\u00f3a cung \u0111\u01b0\u1eddng di chuy\u1ec3n, c\u1ea3nh b\u00e1o \u0111\u00e8o d\u1ed1c v\u00e0 th\u1eddi gian trung chuy\u1ec3n; \"\n                \"k\u1ebft n\u1ed1i c\u00e1c tr\u1ea3i nghi\u1ec7m v\u0103n h\u00f3a gi\u1eefa c\u00e1c t\u1ec9nh th\u00e0nh li\u00ean quan.\"\n            )\n    elif scope == \"INTERNATIONAL_TRAVEL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Cross-Border Scope]: Assist with international flight connections, airport transfers, and customs guidance.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ph\u1ea1m vi Xuy\u00ean qu\u1ed1c gia]: H\u1ed7 tr\u1ee3 th\u00f4ng tin th\u1ecb th\u1ef1c/e-visa, k\u1ebft n\u1ed1i chuy\u1ebfn bay v\u00e0 trung chuy\u1ec3n s\u00e2n bay.\"\n            )\n\n    # 3. Persona Dimension (Family / Backpacker / Solo / Explorer)\n    persona = profile.persona if profile else None\n    if persona == \"FAMILY\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Family Persona]: Prioritize safe, accessible attractions for children and elderly travelers, relaxing pacing, and cozy dining.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ch\u00e2n dung Gia \u0111\u00ecnh]: \u01afu ti\u00ean c\u00e1c \u0111i\u1ec3m tham quan an to\u00e0n, d\u1ec5 ti\u1ebfp c\u1eadn cho ng\u01b0\u1eddi l\u1edbn tu\u1ed5i v\u00e0 tr\u1ebb nh\u1ecf, \"\n                \"nh\u00e0 h\u00e0ng c\u00f3 kh\u00f4ng gian \u1ea5m c\u00fang, l\u1ecbch tr\u00ecnh thong th\u1ea3 tr\u00e1nh qu\u00e1 t\u1ea3i.\"\n            )\n    elif persona == \"BACKPACKER\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Backpacker Persona]: Highlight winding mountain passes, authentic homestays, street food gems, budget optimization, and trekking.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ch\u00e2n dung Ph\u01b0\u1ee3t th\u1ee7 / B\u1ee5i]: \u01afu ti\u00ean tr\u1ea3i nghi\u1ec7m cung \u0111\u01b0\u1eddng \u0111\u00e8o d\u1ed1c, homestay b\u1ea3n \u0111\u1ecba, \u1ea9m th\u1ef1c \u0111\u01b0\u1eddng ph\u1ed1 ng\u00e1ch \"\n                \"v\u00e0 c\u00e1c ho\u1ea1t \u0111\u1ed9ng kh\u00e1m ph\u00e1 m\u1ea1o hi\u1ec3m v\u1eeba s\u1ee9c, t\u1ed1i \u01b0u chi ph\u00ed.\"\n            )\n    elif persona == \"SOLO\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Solo Persona]: Suggest peaceful, secure spaces, scenic coffee spots, personal rejuvenation, and mindful cultural immersion.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ch\u00e2n dung \u0110\u1ed9c h\u00e0nh (Solo)]: G\u1ee3i \u00fd kh\u00f4ng gian y\u00ean t\u0129nh, an to\u00e0n, c\u00e0 ph\u00ea ng\u1eafm c\u1ea3nh v\u00e0 tr\u1ea3i nghi\u1ec7m ch\u1eefa l\u00e0nh, chi\u00eam nghi\u1ec7m v\u0103n h\u00f3a.\"\n            )\n    elif persona == \"EXPLORER\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Explorer Persona]: Prioritize deep cultural heritage, traditional handicraft villages, archaeological history, and indigenous roots.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Ch\u00e2n dung Nh\u00e0 kh\u00e1m ph\u00e1]: \u01afu ti\u00ean chi\u1ec1u s\u00e2u di s\u1ea3n, l\u00e0ng ngh\u1ec1 th\u1ee7 c\u00f4ng c\u1ed5 truy\u1ec1n, l\u1ecbch s\u1eed kh\u1ea3o c\u1ed5 v\u00e0 ngu\u1ed3n g\u1ed1c \u1ea9m th\u1ef1c.\"\n            )\n\n    full_system = base_instructions\n    header_guidance = \"### ADAPTIVE CONSULTATION GUIDELINES (COEXISTING PROFILES):\" if is_english else \"### \u0110\u1ecaNH H\u01af\u1edaNG T\u01af V\u1ea4N TH\u00cdCH \u1ee8NG (COEXISTING PROFILES):\"\n    if guidance_sections:\n        full_system += f\"\\n\\n{header_guidance}\\n\" + \"\\n\".join(guidance_sections)\n\n    if profile:\n        details = []\n        origin_text = profile.origin_province or (profile.origin_location.province if profile.origin_location else None)\n        dest_text = profile.destination or (profile.destination_location.province if profile.destination_location else None)\n        if dest_text: details.append(f\"Destination: {dest_text}\" if is_english else f\"\u0110i\u1ec3m \u0111\u1ebfn: {dest_text}\")\n        if origin_text: details.append(f\"Origin: {origin_text}\" if is_english else f\"Kh\u1edfi h\u00e0nh: {origin_text}\")\n        if profile.budget: details.append(f\"Budget: {profile.budget}\" if is_english else f\"Ng\u00e2n s\u00e1ch: {profile.budget}\")\n        if profile.travel_duration: details.append(f\"Duration: {profile.travel_duration} days\" if is_english else f\"Th\u1eddi gian: {profile.travel_duration} ng\u00e0y\")\n        if profile.interests: details.append(f\"Interests: {', '.join(profile.interests)}\" if is_english else f\"S\u1edf th\u00edch: {', '.join(profile.interests)}\")\n        if details:\n            header_profile = \"### TRIP PROFILE INFORMATION:\" if is_english else \"### TH\u00d4NG TIN H\u1ed2 S\u01a0 H\u00c0NH TR\u00ccNH:\"\n            full_system += f\"\\n\\n{header_profile}\\n- \" + \"\\n- \".join(details)\n\n    if rag_facts_text:\n        header_rag = \"### VERIFIED RETRIEVED FACTS (RAG GROUNDING):\" if is_english else \"### TH\u00d4NG TIN TRA C\u1ee8U \u0110\u00c3 \u0110\u1ed0I SO\u00c1T (RETRIEVED FACTS):\"\n        full_system += f\"\\n\\n{header_rag}\\n{rag_facts_text}\"\n\n    return full_system\n\n# Endpoints\n@app.get(\"/health\")\nasync def health_check():\n    \"\"\"\n    Public health check endpoint returning minimal non-sensitive data.\n    Never exposes GPU diagnostics, VRAM, model inventory, or internal concurrency counter without auth.\n    \"\"\"\n    ollama_ok = False\n    model_ready = False\n    try:\n        async with httpx.AsyncClient(timeout=3.0) as client:\n            res = await client.get(f\"{OLLAMA_BASE_URL}/api/tags\")\n            if res.status_code == 200:\n                ollama_ok = True\n                models = [m.get(\"name\") for m in res.json().get(\"models\", [])]\n                model_ready = MODEL_NAME in models or any(MODEL_NAME in m for m in models)\n    except Exception:\n        pass\n\n    return {\n        \"status\": \"healthy\" if (ollama_ok and model_ready) else \"degraded\" if ollama_ok else \"offline\",\n        \"service\": \"travi-colab-gpu-inference\",\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime())\n    }\n\n@app.get(\"/api/model-info\", response_model=ModelInfoResponse)\nasync def model_info(token: str = Depends(verify_token)):\n    \"\"\"Provides model metadata, quantization details and memory footprint (Protected).\"\"\"\n    ollama_ok = False\n    model_ready = False\n    try:\n        async with httpx.AsyncClient(timeout=3.0) as client:\n            res = await client.get(f\"{OLLAMA_BASE_URL}/api/tags\")\n            if res.status_code == 200:\n                ollama_ok = True\n                models = [m.get(\"name\") for m in res.json().get(\"models\", [])]\n                model_ready = MODEL_NAME in models or any(MODEL_NAME in m for m in models)\n    except Exception:\n        pass\n\n    return ModelInfoResponse(\n        model=MODEL_NAME,\n        quantization=\"Q4_0 (GGUF 4-bit Quantization)\",\n        gpu=gpu_info,\n        keep_alive=\"infinite (-1, model persists in VRAM)\",\n        rag_facts_count=len(rag_corpus),\n        context_window=32768,\n        runtime_expires_note=\"Temporary Google Colab runtime. Endpoint expires when notebook kernel disconnects.\",\n        active_model_ready=model_ready,\n        concurrency_max=MAX_CONCURRENT_REQUESTS\n    )\n\n@app.post(\"/api/chat\")\nasync def chat(request: ChatRequest, token: str = Depends(verify_token)):\n    \"\"\"\n    Main travel AI inference endpoint with Vietnam Tourism RAG grounding,\n    Adaptive Customer Profile Personalization, and concurrency locking.\n    \"\"\"\n    # 1. Retrieve RAG grounding facts\n    retrieved_facts = retrieve_rag_context(request.prompt, top_k=3)\n    rag_context_text = \"\"\n    sources_metadata = []\n\n    if retrieved_facts:\n        for fact in retrieved_facts:\n            rag_context_text += (\n                f\"- T\u00ean: {fact['name']} ({fact['province']})\\n\"\n                f\"  M\u00f4 t\u1ea3: {fact['description']}\\n\"\n                f\"  Tr\u1ea3i nghi\u1ec7m: {', '.join(fact.get('experiences', []))}\\n\"\n                f\"  \u1ea8m th\u1ef1c: {', '.join(fact.get('foods', []))}\\n\"\n                f\"  Th\u1eddi ti\u1ebft: {fact.get('weather_notes', '')}\\n\"\n                f\"  Ngu\u1ed3n \u0111\u1ed1i so\u00e1t: {fact['source']}\\n\"\n            )\n            sources_metadata.append({\n                \"name\": fact[\"name\"],\n                \"province\": fact[\"province\"],\n                \"source\": fact[\"source\"],\n                \"coordinate_status\": fact.get(\"coordinate_status\", \"unverified\")\n            })\n\n    # 2. Build personalized system prompt (Server controlled - Prompt Injection Defense)\n    system_prompt = build_adaptive_system_prompt(request.profile, rag_context_text)\n\n    # 3. Build Ollama messages array\n    messages = [{\"role\": \"system\", \"content\": system_prompt}]\n\n    if request.history:\n        for prev in request.history[-10:]:\n            messages.append({\"role\": prev.role, \"content\": prev.content})\n\n    messages.append({\"role\": \"user\", \"content\": request.prompt})\n\n    # 4. Handle Streaming SSE vs Non-Streaming JSON\n    cust_type = (request.profile.customer_type if request.profile and request.profile.customer_type else \"DOMESTIC\")\n    if request.stream:\n        return StreamingResponse(\n            stream_generator(messages, sources_metadata, cust_type),\n            media_type=\"text/event-stream\"\n        )\n    else:\n        return await non_streaming_generate(messages, sources_metadata, cust_type)\n\nasync def non_streaming_generate(messages: List[Dict[str, str]], sources: List[Dict[str, Any]], customer_type: str):\n    \"\"\"Executes non-streaming completion with concurrency lock and timeout.\"\"\"\n    try:\n        async with asyncio.timeout(REQUEST_TIMEOUT_SECONDS):\n            async with inference_semaphore:\n                async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT_SECONDS) as client:\n                    payload = {\n                        \"model\": MODEL_NAME,\n                        \"messages\": messages,\n                        \"stream\": False,\n                        \"keep_alive\": -1,\n                        \"options\": {\n                            \"temperature\": 0.6,\n                            \"top_p\": 0.9,\n                            \"num_ctx\": 4096\n                        }\n                    }\n                    response = await client.post(f\"{OLLAMA_BASE_URL}/api/chat\", json=payload)\n                    if response.status_code != 200:\n                        raise HTTPException(\n                            status_code=502,\n                            detail=f\"Ollama upstream error: {response.text}\"\n                        )\n                    data = response.json()\n                    content = data.get(\"message\", {}).get(\"content\", \"\")\n\n                    return {\n                        \"status\": \"success\",\n                        \"content\": content,\n                        \"reply\": content,\n                        \"sources\": sources,\n                        \"personalized_for\": customer_type,\n                        \"suggestions\": [\n                            \"T\u00ecm hi\u1ec3u \u1ea9m th\u1ef1c b\u1ea3n \u0111\u1ecba\",\n                            \"Ph\u01b0\u01a1ng ti\u1ec7n di chuy\u1ec3n \u0111\u1ebfn \u0111\u00e2y\",\n                            \"Th\u1eddi \u0111i\u1ec3m l\u00fd t\u01b0\u1edfng nh\u1ea5t trong n\u0103m\"\n                        ],\n                        \"provenance\": {\n                            \"providerName\": f\"Google Colab GPU Qwen2-7B ({gpu_info.get('name', 'T4')})\",\n                            \"isLiveAi\": True,\n                            \"modeDescription\": \"Ollama Serverless GPU Container via Secure HTTPS Tunnel\",\n                            \"vramAllocatedMB\": gpu_info.get(\"total_memory_mb\", 15360)\n                        }\n                    }\n    except asyncio.TimeoutError:\n        raise HTTPException(\n            status_code=504,\n            detail=f\"Inference timed out after {REQUEST_TIMEOUT_SECONDS}s.\"\n        )\n    except Exception as e:\n        logger.error(f\"Inference execution failed: {e}\")\n        raise HTTPException(status_code=500, detail=str(e))\n\nasync def stream_generator(messages: List[Dict[str, str]], sources: List[Dict[str, Any]], customer_type: str):\n    \"\"\"Yields Server-Sent Events (SSE) token chunks as generated by Ollama.\"\"\"\n    try:\n        async with inference_semaphore:\n            async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT_SECONDS) as client:\n                payload = {\n                    \"model\": MODEL_NAME,\n                    \"messages\": messages,\n                    \"stream\": True,\n                    \"keep_alive\": -1,\n                    \"options\": {\n                        \"temperature\": 0.6,\n                        \"top_p\": 0.9,\n                        \"num_ctx\": 4096\n                    }\n                }\n                async with client.stream(\"POST\", f\"{OLLAMA_BASE_URL}/api/chat\", json=payload) as resp:\n                    if resp.status_code != 200:\n                        yield f\"data: {json.dumps({'error': 'Upstream Ollama streaming failed'})}\\n\\n\"\n                        return\n\n                    async for line in resp.aiter_lines():\n                        if not line:\n                            continue\n                        try:\n                            chunk = json.loads(line)\n                            token = chunk.get(\"message\", {}).get(\"content\", \"\")\n                            done = chunk.get(\"done\", False)\n                            event_data = {\n                                \"chunk\": token,\n                                \"done\": done\n                            }\n                            if done:\n                                event_data[\"sources\"] = sources\n                                event_data[\"personalized_for\"] = customer_type\n                                event_data[\"provenance\"] = {\n                                    \"providerName\": f\"Google Colab GPU Qwen2-7B ({gpu_info.get('name', 'T4')})\",\n                                    \"isLiveAi\": True\n                                }\n                            yield f\"data: {json.dumps(event_data)}\\n\\n\"\n                            if done:\n                                break\n                        except Exception:\n                            continue\n    except Exception as err:\n        yield f\"data: {json.dumps({'error': str(err)})}\\n\\n\"\n\nif __name__ == \"__main__\":\n    import uvicorn\n    port = int(os.getenv(\"PORT\", \"8000\"))\n    logger.info(f\"Starting Travi Colab GPU Backend on 127.0.0.1:{port}...\")\n    uvicorn.run(app, host=\"127.0.0.1\", port=port, log_level=\"info\")\n\n")
print("Saved audited app.py with RAG Grounding, Adaptive Profiles, 16KB Guard, and Token Security.")


In [ ]:
# STEP 6: START FASTAPI & LAUNCH SECURE CLOUDFLARE TUNNEL
import threading
import uvicorn
from pycloudflared import try_cloudflare
import app as colab_app

def run_server():
    uvicorn.run("app:app", host="127.0.0.1", port=8000, log_level="warning")

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)

# Launch temporary Cloudflare HTTPS tunnel to expose 127.0.0.1:8000
tunnel = try_cloudflare(port=8000)
public_url = tunnel.tunnel.strip()
session_token = colab_app.COLAB_AI_API_TOKEN

print("=" * 75)
print("🚀 TRAVI GOOGLE COLAB GPU INFERENCE BACKEND IS ONLINE!")
print(f"Public HTTPS Endpoint: {public_url}")
print(f"API Token saved to: {colab_app.TOKEN_FILE_PATH} (mode 0600)")
print(f"Health Check URL (Public): {public_url}/health")
print(f"Model Info URL (Auth Required): {public_url}/api/model-info")
print(f"Chat Endpoint (Auth Required): {public_url}/api/chat")
print("=" * 75)
print("CONFIGURING YOUR TRAVI SERVER (.env):")
print(f"COLAB_AI_BASE_URL={public_url}")
print(f"COLAB_AI_API_TOKEN={session_token}")
print("COLAB_AI_ENABLED=true")
print("=" * 75)


In [ ]:
# STEP 7: AUTOMATED VERIFICATION SELF-TESTS
import httpx
headers = {"Authorization": f"Bearer {colab_app.COLAB_AI_API_TOKEN}"}
client = httpx.Client(base_url="http://127.0.0.1:8000", headers=headers, timeout=35.0)

# 1. Health check (Public minimal non-sensitive data)
h = httpx.get("http://127.0.0.1:8000/health").json()
print("1. Health Check:", h["status"], "Service:", h["service"])
assert h["status"] in ["healthy", "degraded"]
assert "gpu" not in h, "Security failure: Public health check leaked GPU data"

# 2. Auth guard
unauth = httpx.get("http://127.0.0.1:8000/api/model-info")
print("2. Auth Guard Status:", unauth.status_code)
assert unauth.status_code == 401

# 3. Model Info with auth
info = client.get("/api/model-info").json()
print("3. Model Info:", info["model"], "Quantization:", info["quantization"], "GPU:", info["gpu"]["name"])
assert info["model"] == "qwen2:7b"

# 4. Adaptive Chat - International Profile (English)
intl_req = {
    "prompt": "I am visiting Hanoi for 3 days. What temple etiquette should I follow?",
    "profile": {"market": "INTERNATIONAL", "language": "EN", "origin_country": "France", "destination": "Hà Nội"}
}
ans_intl = client.post("/api/chat", json=intl_req).json()
print("4. International Profile Reply Preview:", ans_intl["content"][:120] + "...")
assert ans_intl["status"] == "success"

# 5. Adaptive Chat - Local Profile (Vietnamese)
local_req = {
    "prompt": "Cuối tuần này ở Mộc Châu có điểm cắm trại nào vắng người?",
    "profile": {"travel_scope": "LOCAL", "persona": "FAMILY", "origin_province": "Sơn La", "destination": "Mộc Châu"}
}
ans_local = client.post("/api/chat", json=local_req).json()
print("5. Local Profile Reply Preview:", ans_local["content"][:120] + "...")
assert ans_local["status"] == "success"

# 6. Payload size limit guard (16KB max)
large_payload = {"prompt": "A" * 20000}
res_large = client.post("/api/chat", json=large_payload)
print("6. Large payload rejection status:", res_large.status_code)
assert res_large.status_code in [413, 422]

print("\nALL COLAB BACKEND VERIFICATION TESTS PASSED!")


In [ ]:
# STEP 7: AUTOMATED VERIFICATION SELF-TESTS
import httpx
headers = {"Authorization": f"Bearer {colab_app.COLAB_AI_API_TOKEN}"}
client = httpx.Client(base_url="http://127.0.0.1:8000", headers=headers, timeout=30.0)

# 1. Health check
h = client.get("/health").json()
print("1. Health Check:", h["status"], "GPU:", h["gpu"]["name"])
assert h["status"] == "healthy"

# 2. Auth guard
unauth = httpx.get("http://127.0.0.1:8000/api/model-info")
print("2. Auth Guard Status:", unauth.status_code)
assert unauth.status_code == 401

# 3. Model Info with auth
info = client.get("/api/model-info").json()
print("3. Model Info:", info["model"], "Quantization:", info["quantization"])

# 4. Adaptive Chat - International Profile
intl_req = {
    "prompt": "I am traveling to Hanoi. What temple etiquette should I follow?",
    "profile": {"customer_type": "INTERNATIONAL", "origin_country": "France", "destination": "Hà Nội"}
}
ans_intl = client.post("/api/chat", json=intl_req).json()
print("4. International Profile Reply Preview:", ans_intl["content"][:150] + "...")
assert ans_intl["personalized_for"] == "INTERNATIONAL"

# 5. Adaptive Chat - Local Profile
local_req = {
    "prompt": "Có quán ăn nào yên tĩnh ở Mộc Châu không?",
    "profile": {"customer_type": "LOCAL", "origin_province": "Sơn La", "destination": "Mộc Châu"}
}
ans_local = client.post("/api/chat", json=local_req).json()
print("5. Local Profile Reply Preview:", ans_local["content"][:150] + "...")
assert ans_local["personalized_for"] == "LOCAL"

print("\nALL COLAB BACKEND TESTS PASSED!")
